# 🏥 Task 3: Hyperparameter Tuning với Sweep Job

## Mục tiêu - Milestone 4
Dùng **Azure ML Sweep Job** để tự động tìm giá trị `reg_rate` tốt nhất cho Logistic Regression.

## Ý tưởng
```
Sweep Job = chạy nhiều Command Job song song,
            mỗi job với reg_rate khác nhau,
            chọn job có accuracy cao nhất.
```

## Cấu trúc
```
Task3-Hyperparameter/
├── Task3_Hyperparameter_Sweep.ipynb  ← File này
└── src/
    ├── ER_Triage_Dataset.csv
    └── train.py                      ← Script log training_accuracy_score
```

## Bước 1: Setup

In [ ]:
pip show azure-ai-ml

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
    print("✅ DefaultAzureCredential OK")
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
print(f"✅ Workspace: {ml_client.workspace_name}")

## Bước 2: Đăng ký Dataset dạng URI_FILE

> Sweep Job dùng `uri_file` (không dùng MLTable như Task 1)

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Đăng ký URI_FILE data asset
my_data = Data(
    path="./src/ER_Triage_Dataset.csv",
    type=AssetTypes.URI_FILE,
    description="ER Triage Dataset - URI File cho Command/Sweep Jobs",
    name="ER_Triage_File_Dataset",
    version="1"
)

ml_client.data.create_or_update(my_data)
print("✅ Đã đăng ký: ER_Triage_File_Dataset v1 (uri_file)")

## Bước 3: Tạo và test Command Job cơ bản

**Luôn test command job trước** khi chạy sweep.

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

# Command Job với reg_rate cố định = 0.01
job = command(
    code="./src",
    command="python train.py --training_data ${{inputs.ERTriage_data}} --reg_rate ${{inputs.reg_rate}}",
    inputs={
        "ERTriage_data": Input(
            type=AssetTypes.URI_FILE,
            path="azureml:ER_Triage_File_Dataset:1"
        ),
        "reg_rate": 0.01
    },
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute="aml-cluster",
    display_name="ERTriage-sweep-baseline",
    experiment_name="ERTriage-training",
    tags={"model_type": "LogisticRegression"}
)

# Submit command job test
returned_job = ml_client.jobs.create_or_update(job)
print(f"🚀 Baseline Command Job submitted")
print(f"   URL: {returned_job.studio_url}")

In [ ]:
# Chờ baseline job
ml_client.jobs.stream(returned_job.name)
print("✅ Baseline job hoàn thành - script hoạt động tốt!")

## Bước 4: Định nghĩa Search Space

Sẽ thử 3 giá trị `reg_rate`: **0.01, 0.1, 1**

In [ ]:
from azure.ai.ml.sweep import Choice

# Chuyển Command Job thành Sweep-ready bằng cách override inputs
command_job_for_sweep = job(
    reg_rate=Choice(values=[0.001, 0.01, 0.1, 1, 10])
)

print("✅ Search space đã định nghĩa:")
print("   reg_rate ∈ {0.001, 0.01, 0.1, 1, 10}")

## Bước 5: Cấu hình và chạy Sweep Job

In [ ]:
# Cấu hình Sweep Job
sweep_job = command_job_for_sweep.sweep(
    compute="aml-cluster",
    sampling_algorithm="grid",           # Grid search: thử TẤT CẢ giá trị
    primary_metric="training_accuracy_score",  # Metric để so sánh
    goal="Maximize",                      # Mục tiêu: tối đa accuracy
)

sweep_job.experiment_name = "sweep-ERTriage"
sweep_job.display_name = "ERTriage-hyperparameter-sweep"

# Giới hạn
sweep_job.set_limits(
    max_total_trials=5,         # Tổng số trials
    max_concurrent_trials=2,    # Chạy song song 2 trials
    timeout=7200                # Timeout 2 giờ
)

print("✅ Sweep Job cấu hình:")
print(f"   Sampling: Grid Search")
print(f"   Primary metric: training_accuracy_score (Maximize)")
print(f"   Max trials: 5 | Concurrent: 2")
print(f"   Search space: reg_rate ∈ {{0.001, 0.01, 0.1, 1, 10}}")

In [ ]:
# Submit Sweep Job
returned_sweep_job = ml_client.create_or_update(sweep_job)
aml_url = returned_sweep_job.studio_url
print(f"🚀 Sweep Job submitted!")
print(f"   Theo dõi tại: {aml_url}")
print(f"\n⏳ Đang chạy {5} trials song song...")

In [ ]:
# Chờ sweep hoàn thành
ml_client.jobs.stream(returned_sweep_job.name)
print("✅ Sweep Job hoàn thành!")

## Bước 6: Phân tích kết quả Sweep

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt

# Lấy tất cả runs trong sweep experiment
exp = mlflow.get_experiment_by_name("sweep-ERTriage")
runs_df = mlflow.search_runs(
    exp.experiment_id,
    order_by=["metrics.training_accuracy_score DESC"]
)

# Hiển thị kết quả
result_cols = [
    'run_id',
    'params.Regularization rate',
    'metrics.training_accuracy_score',
    'metrics.Recall',
    'metrics.F1_Score',
    'metrics.AUC'
]
available = [c for c in result_cols if c in runs_df.columns]

print("📊 Kết quả Hyperparameter Sweep (sắp xếp theo Accuracy giảm dần):")
print(runs_df[available].to_string(index=False))

In [ ]:
# Biểu đồ so sánh reg_rate vs metrics
if 'params.Regularization rate' in runs_df.columns:
    plot_df = runs_df[available].dropna()
    plot_df['reg_rate'] = pd.to_numeric(plot_df['params.Regularization rate'])
    plot_df = plot_df.sort_values('reg_rate')

    fig, ax = plt.subplots(figsize=(9, 5))

    metrics_to_plot = [
        ('metrics.training_accuracy_score', 'Accuracy', '#2196F3'),
        ('metrics.Recall', 'Recall', '#FF5722'),
        ('metrics.F1_Score', 'F1-Score', '#4CAF50'),
        ('metrics.AUC', 'AUC', '#9C27B0'),
    ]

    for col, label, color in metrics_to_plot:
        if col in plot_df.columns:
            ax.plot(plot_df['reg_rate'], pd.to_numeric(plot_df[col]),
                    'o-', label=label, color=color, linewidth=2, markersize=8)

    ax.set_xscale('log')
    ax.set_xlabel('Regularization Rate (log scale)', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Hyperparameter Sweep: reg_rate vs Metrics', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.savefig('sweep_results.png', dpi=100)
    plt.show()
    print("✅ Đã lưu biểu đồ kết quả")

In [ ]:
# Tìm best trial
best_run = runs_df.iloc[0] if not runs_df.empty else None

if best_run is not None:
    best_reg = best_run.get('params.Regularization rate', 'N/A')
    best_acc = best_run.get('metrics.training_accuracy_score', 'N/A')
    best_auc = best_run.get('metrics.AUC', 'N/A')
    best_recall = best_run.get('metrics.Recall', 'N/A')

    print("🏆 Best Hyperparameter:")
    print(f"   reg_rate = {best_reg}")
    print(f"   Accuracy = {best_acc:.4f}" if isinstance(best_acc, float) else f"   Accuracy = {best_acc}")
    print(f"   AUC      = {best_auc:.4f}" if isinstance(best_auc, float) else f"   AUC      = {best_auc}")
    print(f"   Recall   = {best_recall:.4f}" if isinstance(best_recall, float) else f"   Recall   = {best_recall}")

## 📋 Kết luận Task 3 (Milestone 4)

### Sweep Job đã thực hiện:
| reg_rate | Accuracy | Recall | AUC |
|----------|---------|--------|-----|
| 0.001 | ... | ... | ... |
| 0.01 | ... | ... | ... |
| 0.1 | ... | ... | ... |
| 1.0 | ... | ... | ... |
| 10.0 | ... | ... | ... |

*(Điền kết quả từ sweep vào bảng)*

### Sampling Algorithms:
- **Grid**: Thử tất cả giá trị — dùng khi không gian nhỏ ✅
- **Random**: Ngẫu nhiên — nhanh hơn với không gian lớn
- **Bayesian**: Thông minh, dùng kết quả trước — hiệu quả nhất

### Ghi nhớ y tế:
> reg_rate cao → mô hình đơn giản → có thể bỏ sót bệnh nặng (False Negative)
> Chọn reg_rate cân bằng **Accuracy** VÀ **Recall**!